# Neuron Flow — เริ่มทดลองบน Colab

1. รันเซลล์อัปโหลด แล้วเลือกไฟล์ `neuron_flow_v0.1.0.zip`
2. รันตัวอย่าง ระบบจะแสดงผลตรวจยอดรวมและตำแหน่งรายงาน
3. รันชุดทดสอบเพื่อดูการจัดการ Agent ค้าง/ล้มและการทำงานต่อ
4. ดาวน์โหลดผลกลับเครื่องเมื่อเสร็จ เพราะไฟล์ใน Colab เป็นพื้นที่ชั่วคราว

ระบบใช้ Python standard library ไม่ต้องใช้ API key การทดสอบนี้เป็น workflow ที่กำหนดชัดเจน ยังไม่มีการเรียนรู้เองหรืออ่านภาพ/PDF


In [ ]:
from google.colab import files
from pathlib import Path
import json
import subprocess
import sys
import zipfile

uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.endswith('.zip')]
if len(archives) != 1:
    raise ValueError('เลือก ZIP ของ Neuron Flow เพียงไฟล์เดียว')
unpack = Path('/content/neuron_flow_project')
unpack.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for name in archive.namelist():
        if not (unpack / name).resolve().is_relative_to(unpack.resolve()):
            raise ValueError('พบเส้นทางที่ไม่ถูกต้องใน ZIP')
    archive.extractall(unpack)
PROJECT = unpack / 'neuron_flow'
if not (PROJECT / 'neuron_flow' / '__main__.py').is_file():
    raise ValueError('โครงสร้าง ZIP ไม่ตรงกับ Neuron Flow')
print('พร้อมรัน:', PROJECT)


## รันระบบ

ผลที่ถูกต้องคือ `SUCCEEDED`, ยอดรวม `310.70`, `verified` เป็น `True` และทุกขั้นพยายามเพียงครั้งเดียว


In [ ]:
command = [sys.executable, '-m', 'neuron_flow', 'demo']
completed = subprocess.run(command, cwd=PROJECT, capture_output=True, text=True, timeout=60)
print(completed.stderr)
completed.check_returncode()
result = json.loads(completed.stdout)
print('สถานะ:', result['status'])
print('ผลตรวจ:', result['steps']['verify']['output'])
print('รายงาน:', result['receipt']['path'])
print('จำนวนครั้ง:', {name: step['attempts'] for name, step in result['steps'].items()})


## ตรวจสอบซ้ำ

รันทดสอบพฤติกรรมทั้งหมด รวมการหยุด process ที่ค้างจริง และฆ่า controller แล้วเปิดใหม่
เซลล์นี้อาจใช้เวลามากกว่าตัวอย่างปกติ ไม่ต้องแก้โค้ดระหว่างรัน


In [ ]:
checked = subprocess.run(
    [sys.executable, 'tools/verify.py'], cwd=PROJECT,
    capture_output=True, text=True, timeout=300
)
print(checked.stdout)
print(checked.stderr)
checked.check_returncode()
print((PROJECT / 'verification' / 'VERIFICATION_TH.md').read_text(encoding='utf-8'))


## ทดลองแก้รายการ

แก้ `document` แล้วรันเซลล์นี้ การสร้าง job id จากเนื้อหาช่วยให้ข้อมูลที่เปลี่ยนได้งานใหม่ และข้อมูลเดิมกลับมาใช้ผลเดิม
ราคาต้องเป็นข้อความ เช่น `"25.50"`


In [ ]:
document = {
    'document_id': 'MY-001',
    'currency': 'THB',
    'items': [
        {'description': 'ตัวอย่างสินค้า', 'quantity': 4, 'unit_price': '25.50'},
        {'description': 'อีกรายการ', 'quantity': 2, 'unit_price': '10.00'},
    ],
}
source = PROJECT / 'my_invoice.json'
source.write_text(json.dumps(document, ensure_ascii=False), encoding='utf-8')
custom = subprocess.run(
    [sys.executable, '-m', 'neuron_flow', 'invoice', str(source)],
    cwd=PROJECT, capture_output=True, text=True, timeout=60
)
print(custom.stdout)
print(custom.stderr)
custom.check_returncode()


## ดาวน์โหลดผลลัพธ์

ไฟล์ที่ดาวน์โหลดรวมสถานะงาน รายงาน และผลทดสอบ เก็บไว้เพื่อทำต่อในครั้งถัดไป


In [ ]:
import shutil
backup = shutil.make_archive('/content/neuron_flow_results', 'zip', root_dir=PROJECT)
files.download(backup)
